# TaskMate — Validation Notebook 03: Structured Tool Calling (Nemotron)

**Purpose:** Verify tool declaration schemas, structured tool calling with Nemotron, parameter extraction, and execution through TaskTool and ToolRegistry.

**Operational Note:** This notebook is strictly for isolated experimentation, learning, and verification. It is **NOT** the production runtime.

## 1. Setup Environment and Imports

In [ ]:
import os
import sys
import json

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from backend.app.agent.schemas import ALL_TOOL_SCHEMAS, TASK_TOOL_SCHEMAS
from backend.app.agent.tool_registry import ToolRegistry
from backend.app.core.config import get_settings
from backend.app.services.llm_service import LLMService

registry = ToolRegistry()
schemas = registry.get_tool_schemas()
print(f"Total Registered Tools: {len(schemas)}")
for s in schemas:
    print(f" - {s['function']['name']}: {s['function']['description'][:60]}...")

## 2. Inspect Approved Tool Schemas (create_task, calculate, etc.)

In [ ]:
# Print schema for create_task
create_schema = next(s for s in schemas if s['function']['name'] == 'create_task')
print(json.dumps(create_schema, indent=2))

# Verify user_id is NOT in the exposed schema
assert 'user_id' not in create_schema['function']['parameters']['properties'], "Security violation: user_id exposed to LLM!"
print("✅ Security check passed: user_id is properly hidden from LLM schemas!")

## 3. Test Tool Registry Direct Execution (TaskTool & Utility Tools)

In [ ]:
# Test calculator utility
calc_result = registry.execute_tool("calculate", {"expression": "(20 + 5) * 4"}, user_id="demo_user")
print("Calculator Result:", calc_result)
assert calc_result["success"] is True and calc_result["result"] == 100.0

# Test datetime utility
dt_result = registry.execute_tool("get_date_time", {"query": "tomorrow"}, user_id="demo_user")
print("Datetime Result:", dt_result)
assert dt_result["success"] is True

print("✅ Utility tools execution verified!")

## 4. Test Complete Conversation Flow (End-to-End with Nemotron)

> Note: Requires `NVIDIA_API_KEY` or `LLM_API_KEY`.

In [ ]:
service = LLMService(tool_registry=registry)
user_id = "colab_demo_user"
prompt = "What is 450 * 3?"

try:
    outcome = service.run_conversation_turn(user_id=user_id, user_prompt=prompt)
    print("Assistant Final Response:", outcome["response"])
    print("Executed Tools:", outcome["tool_calls"])
    print("Tool Results:", outcome["tool_results"])
except Exception as err:
    print(f"Flow output/exception (check credentials if offline): {err}")